# Hands-On 4: Build Sequential and Hierarchical Crews

### Definition
A **Crew** coordinates agents and tasks.

- **Sequential:** tasks run in a fixed order and later tasks can use earlier outputs.
- **Hierarchical:** a manager agent coordinates the other agents and decides how work is delegated.

### Goal
Build both patterns, execute them, and compare the execution flow.

In [1]:
from crewai import Agent, Task, Crew, Process
import os
from dotenv import load_dotenv

load_dotenv(r"D:\Agent_Flow\.env", override=True)

print("OpenRouter key loaded:", bool(os.getenv("OPENROUTER_API_KEY")))
print("Model:", os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini"))
print("Base URL:", os.getenv("OPENAI_BASE_URL"))

OpenRouter key loaded: True
Model: openai/gpt-4o-mini
Base URL: https://openrouter.ai/api/v1


In [2]:
llm = os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini")

print("LLM configured:", llm)

LLM configured: openai/gpt-4o-mini


## 1. Create Agents

We use three specialized agents:

- **Researcher** → gathers information
- **Writer** → creates content
- **Editor** → improves the content

For the hierarchical crew, a **Manager** coordinates the team.

In [3]:
researcher = Agent(
    role="Research Specialist",
    goal="Gather accurate and useful information",
    backstory="An expert researcher who produces clear, reliable findings.",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

writer = Agent(
    role="Content Writer",
    goal="Create clear and engaging content",
    backstory="A professional writer who turns research into useful content.",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

editor = Agent(
    role="Content Editor",
    goal="Improve clarity, accuracy, and quality",
    backstory="A meticulous editor who prepares content for publication.",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("Agents created successfully")

Agents created successfully


## 2. Sequential Crew

**Flow:** Research → Write → Edit

In [4]:
research_task = Task(
    description="Research the top 5 benefits of remote work for companies in 2025.",
    expected_output="Five specific benefits with short supporting details.",
    agent=researcher
)

write_task = Task(
    description="Write a concise blog post using the research provided.",
    expected_output="A clear and well-structured blog post.",
    agent=writer,
    context=[research_task]
)

edit_task = Task(
    description="Edit the blog post for clarity, grammar, and accuracy.",
    expected_output="A polished final blog post.",
    agent=editor,
    context=[write_task]
)

sequential_crew = Crew(
    agents=[researcher, writer, editor],
    tasks=[research_task, write_task, edit_task],
    process=Process.sequential,
    verbose=True,
    memory=False
)

print("Sequential crew created")

Sequential crew created


In [6]:
print("=" * 50)
print("RUNNING SEQUENTIAL CREW")
print("=" * 50)

seq_result = await sequential_crew.kickoff_async()

print("SEQUENTIAL OUTPUT:")
print(seq_result)

RUNNING SEQUENTIAL CREW


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 415b52c9-3edf-48b5-9009-bba8f35e1d5d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the top 5 benefits of remote work for companies in 2025.                                        │
│  ID: 28181d95-3916-4dc8-81a3-878feea2a7e8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Task: Research the top 5 benefits of remote work for companies in 2025.                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  As of 2025, the trend of remote work continues to evolve, and companies are reaping numerous benefits from     │
│  facilitating flexible work arrangements. Here are the top five benefits of remote work for companies:          │
│                                                                                                                 │
│  1. **Cost Savings**                                                                                            │
│     Remote work significantly reduces overhead costs for companies. Businesses can save on office space,        │
│  utilities, and other facilities-related expenses. With fewer employees in a physical office, companies can     │
│  downsize their real estate footprint, leading to substantial savings that can be reinvested into other areas   │
│  of the business.                                                                                               │
│                                                                                                                 │
│  2. **Access to a Wider Talent Pool**                                                                           │
│     By allowing remote work, companies can recruit talent from a broader geographic area rather than limiting   │
│  themselves to local candidates. This access to a diverse talent pool can enhance the overall skill set within  │
│  the company, allowing organizations to find employees who fit their culture and possess specialized skills     │
│  that may not be available locally.                                                                             │
│                                                                                                                 │
│  3. **Increased Employee Productivity**                                                                         │
│     Many studies have shown that remote workers often report higher productivity levels. The flexibility of     │
│  working from home allows employees to create their own ideal working environments and schedules, leading to    │
│  fewer distractions and interruptions often found in traditional office settings. This autonomy can contribute  │
│  to improved focus and efficiency in completing tasks.                                                          │
│                                                                                                                 │
│  4. **Improved Employee Well-Being**                                                                            │
│     Remote work can greatly enhance employee satisfaction and overall well-being. With the flexibility to       │
│  balance personal and professional responsibilities, employees are less likely to experience burnout. The       │
│  reduction in commuting time also allows for more personal time, leading to higher morale and retention rates   │
│  as employees feel valued and trusted by their employers.                                                       │
│                                                                                                                 │
│  5. **Business Continuity and Resilience**                                                                      │
│     Embracing remote work enables companies to maintain operations during disruptions, such as natural          │
│  disasters, pandemics, or other unforeseen events. A re

... (10 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Edit the blog post for clarity, grammar, and accuracy.                                                   │
│  Agent: Content Editor                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 415b52c9-3edf-48b5-9009-bba8f35e1d5d                                                                       │
│  Final Output: # Embracing the Future: The Top Five Benefits of Remote Work for Companies in 2025               │
│                                                                                                                 │
│  As we venture further into 2025, the trend of remote work continues to reshape the corporate landscape,        │
│  offering innovative solutions that benefit both companies and their employees. Flexible work arrangements are  │
│  not just a fleeting response to recent global challenges; they've become a strategic necessity that            │
│  organizations are embracing for a multitude of reasons. Here’s a closer look at the top five benefits of       │
│  remote work for businesses today.                                                                              │
│                                                                                                                 │
│  ### 1. Cost Savings                                                                                            │
│                                                                                                                 │
│  One of the most immediate advantages of remote work is the significant cost savings for companies. By          │
│  reducing the need for extensive office space and related expenses—such as utilities, maintenance, and          │
│  supplies—organizations can optimize their budgets and streamline operations. This transition allows            │
│  businesses to downsize their physical real estate footprint, resulting in substantial savings that can be      │
│  reinvested in growth, technology, or employee development.                                                     │
│                                                                                                                 │
│  ### 2. Access to a Wider Talent Pool                                                                           │
│                                                                                                                 │
│  Remote work opens the door to a broader and more diverse talent pool. Companies are no longer restricted to    │
│  hiring local candidates; they can seek out the best talent from anywhere in the world. This not only enhances  │
│  the overall skill set within the organization but also fosters a diverse workforce. A diverse team brings      │
│  different perspectives and ideas, leading to greater creativity and innovation, which ultimately contributes   │
│  to the company's success.                                                                                      │
│                                                                                                                 │
│  ### 3. Increased Employee Productivity                                                                         │
│                                                                                                                 │
│  Numerous studies have shown that remote workers often report higher levels of productivity. The flexibility    │
│  to tailor their work environments and schedules leads to fewer distractions and interruptions typically found  │
│  in traditional office settings. Employees can choose to work during their most productive hours, and many      │
│  find that the comfort of home enhances their focus an

SEQUENTIAL OUTPUT:
# Embracing the Future: The Top Five Benefits of Remote Work for Companies in 2025

As we venture further into 2025, the trend of remote work continues to reshape the corporate landscape, offering innovative solutions that benefit both companies and their employees. Flexible work arrangements are not just a fleeting response to recent global challenges; they've become a strategic necessity that organizations are embracing for a multitude of reasons. Here’s a closer look at the top five benefits of remote work for businesses today.

### 1. Cost Savings

One of the most immediate advantages of remote work is the significant cost savings for companies. By reducing the need for extensive office space and related expenses—such as utilities, maintenance, and supplies—organizations can optimize their budgets and streamline operations. This transition allows businesses to downsize their physical real estate footprint, resulting in substantial savings that can be reinvested i

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 3. Hierarchical Crew

**Flow:**

```text
Manager
   ↓
Research / Writing / Editing
   ↓
Manager reviews and coordinates
   ↓
Final result
```

The manager controls task delegation.

In [7]:
researcher_h = Agent(
    role="Research Specialist",
    goal="Gather accurate and useful information",
    backstory="An expert researcher who produces reliable findings.",
    llm=llm,
    verbose=True,
    allow_delegation=True
)

writer_h = Agent(
    role="Content Writer",
    goal="Create clear and engaging content",
    backstory="A professional writer who turns research into content.",
    llm=llm,
    verbose=True,
    allow_delegation=True
)

editor_h = Agent(
    role="Content Editor",
    goal="Improve clarity and final quality",
    backstory="A meticulous editor focused on publication quality.",
    llm=llm,
    verbose=True,
    allow_delegation=True
)

manager = Agent(
    role="Project Manager",
    goal="Coordinate the team and deliver a high-quality final result",
    backstory="An experienced manager who delegates work, reviews outputs, and coordinates the team.",
    llm=llm,
    verbose=True,
    allow_delegation=True
)

print("Hierarchical agents created")

Hierarchical agents created


In [8]:
research_task_h = Task(
    description="Research the top 5 benefits of remote work for companies in 2025.",
    expected_output="Five specific benefits with supporting details."
)

write_task_h = Task(
    description="Create a concise blog post about the benefits of remote work using the available research.",
    expected_output="A clear and engaging blog post."
)

edit_task_h = Task(
    description="Review and improve the final content for accuracy, clarity, and readability.",
    expected_output="A polished final blog post."
)

hierarchical_crew = Crew(
    agents=[researcher_h, writer_h, editor_h],
    tasks=[research_task_h, write_task_h, edit_task_h],
    process=Process.hierarchical,
    manager_agent=manager,
    verbose=True,
    memory=False
)

print("Hierarchical crew created")

Hierarchical crew created


In [9]:
print("=" * 50)
print("RUNNING HIERARCHICAL CREW")
print("=" * 50)

hier_result = await hierarchical_crew.kickoff_async()

print("HIERARCHICAL OUTPUT:")
print(hier_result)

RUNNING HIERARCHICAL CREW


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2ab1e654-d7e9-4b2b-af88-e796e007ff0e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the top 5 benefits of remote work for companies in 2025.                                        │
│  ID: f2c75149-cd78-451c-84f3-a1494c16ab12                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Project Manager                                                                                         │
│                                                                                                                 │
│  Task: Research the top 5 benefits of remote work for companies in 2025.                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': 'Research the top 5 benefits of remote work for companies in 2025, including specific details   │
│  for each benefit.', 'context': "The goal is to identify and elaborate on five major advantages th...           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Specialist                                                                                     │
│                                                                                                                 │
│  Task: Research the top 5 benefits of remote work for companies in 2025, including specific details for each    │
│  benefit.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (28 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review and improve the final content for accuracy, clarity, and readability.                             │
│  Agent: Project Manager                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2ab1e654-d7e9-4b2b-af88-e796e007ff0e                                                                       │
│  Final Output: **The Advantages of Remote Work for Companies in 2025**                                          │
│                                                                                                                 │
│  As we look ahead to 2025, the landscape of work continues to evolve dramatically. One of the most significant  │
│  shifts we have witnessed is the increasing adoption of remote work arrangements. Companies that embrace this   │
│  model are discovering a wealth of advantages that not only benefit their bottom line but also enhance the      │
│  experience of their employees. Here, we explore five key benefits of remote work that can transform the        │
│  workplace for organizations in the coming years.                                                               │
│                                                                                                                 │
│  ### 1. Increased Productivity                                                                                  │
│                                                                                                                 │
│  Remote work allows employees to create environments that foster their individual productivity. Without the     │
│  distractions of a traditional office setting—such as frequent meetings or casual interruptions—many workers    │
│  find that they can focus more effectively on their tasks. Studies have shown that remote employees often work  │
│  longer hours and report higher levels of engagement, contributing to increased output. By removing the         │
│  barriers of commuting and rigid office hours, companies can harness the full potential of their workforce.     │
│                                                                                                                 │
│  ### 2. Cost Savings                                                                                            │
│                                                                                                                 │
│  Transitioning to remote work can lead to significant cost savings for companies. With fewer employees in the   │
│  office, businesses can downsize their physical spaces, reducing rent and overhead costs. Additionally,         │
│  expenses related to utilities, office supplies, and facilities maintenance can also decrease. With reduced     │
│  operational costs, companies can redirect those funds toward other strategic investments, such as innovative   │
│  technologies or employee development.                                                                          │
│                                                                                                                 │
│  ### 3. Access to a Wider Talent Pool                                                                           │
│                                                                                                                 │
│  One of the most exciting advantages of remote work is the ability to tap into a global talent pool. Companies  │
│  are no longer restricted to hiring candidates within a specific geographic area. This access allows            │
│  businesses to find individuals with the right skills and experience, regardless of where they live. Moreover,  │
│  a diverse workforce brings a variety of perspectives 

HIERARCHICAL OUTPUT:
**The Advantages of Remote Work for Companies in 2025**

As we look ahead to 2025, the landscape of work continues to evolve dramatically. One of the most significant shifts we have witnessed is the increasing adoption of remote work arrangements. Companies that embrace this model are discovering a wealth of advantages that not only benefit their bottom line but also enhance the experience of their employees. Here, we explore five key benefits of remote work that can transform the workplace for organizations in the coming years.

### 1. Increased Productivity

Remote work allows employees to create environments that foster their individual productivity. Without the distractions of a traditional office setting—such as frequent meetings or casual interruptions—many workers find that they can focus more effectively on their tasks. Studies have shown that remote employees often work longer hours and report higher levels of engagement, contributing to increased output. 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 4. Compare

| Sequential | Hierarchical |
|---|---|
| Fixed task order | Manager controls execution |
| Simple and predictable | Flexible and dynamic |
| Lower overhead | More manager/LLM overhead |
| Best for dependent pipelines | Best for complex coordination |

### Key difference

**Sequential:** `Task 1 → Task 2 → Task 3`

**Hierarchical:** `Manager → delegates/reviews → final result`

In [10]:
print("=== HANDS-ON 4 COMPLETE ===")
print("✓ Multiple agents")
print("✓ Dependent tasks")
print("✓ Sequential crew")
print("✓ Sequential execution")
print("✓ Hierarchical crew")
print("✓ Hierarchical execution")
print("✓ Execution pattern comparison")

=== HANDS-ON 4 COMPLETE ===
✓ Multiple agents
✓ Dependent tasks
✓ Sequential crew
✓ Sequential execution
✓ Hierarchical crew
✓ Hierarchical execution
✓ Execution pattern comparison
